In [2]:
# pip install pdfplumber

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 20.7 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 17.5 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 19.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [pdfplumber]4 [pdfminer.six]

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [9]:
import pdfplumber
import json
import re

SKIP_PAGES  = {1, 8, 72, 73, 74, 75, 76}
PAGE_NUM_RE = re.compile(r'^\s*-\s*\d+\s*-\s*$')


def extract_raw(pdf_path: str) -> dict:
    """
    One entry per page. Each entry is just the page number and its
    cleaned text as a single string. No chunking, no tagging, no parsing.
    Only artifact removed is the page number header line.
    """
    pages = []

    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages, start=1):
            if page_num in SKIP_PAGES:
                continue

            raw_text = page.extract_text() or ''

            # Only cleaning we do — strip page number artifact and empty lines
            lines = []
            for line in raw_text.split('\n'):
                stripped = line.strip()
                if PAGE_NUM_RE.match(stripped):
                    continue
                if not stripped:
                    continue
                lines.append(stripped)

            page_text = '\n'.join(lines)

            if page_text:
                pages.append({
                    'page':  page_num,
                    'text':  page_text
                })

    return {
        'source':      pdf_path,
        'total_pages': len(pages),
        'pages':       pages
    }


if __name__ == '__main__':
    raw = extract_raw('Official-2025-26-NBA-Playing-Rules.pdf')

    with open('nba_rules_raw.json', 'w', encoding='utf-8') as f:
        json.dump(raw, f, indent=2, ensure_ascii=False)

    print(f"Done — {raw['total_pages']} pages written to nba_rules_raw.json")

Done — 69 pages written to nba_rules_raw.json


Done — 403 chunks written to nba_rules_naive_rag.json
